# CS 483/583 · Module 11 · Sequence Modeling, Attention, and Transformers

Suppose a display shows red, blue, red in that order. At the last position, how could a model decide which earlier signals to use? Build a small attention layer to see exactly what each position contributes.

Module 10 turned items into embedding vectors and compared them with a dot product. We will keep one vector per occurrence in a sequence, then use similarity to decide how much information to take from each occurrence.

## Running the examples

Use a Python 3 notebook with PyTorch installed. A CPU is sufficient. Run the cells in order, then change an input and predict what will happen.

If both occurrences of red use the same embedding, which number in that embedding tells us which red came first?

<details><summary>Reveal answer</summary>

Neither occurrence carries its position in the shared color embedding. We must preserve the row order and give the model some way to use it.

</details>

## Keep an ordered row for each occurrence

A token is one unit we choose to represent, such as a color signal. An embedding assigns each token a vector of numbers. Place the vectors in reading order. A row index helps us find a token in code. Later we will give the attention calculation access to position too.

Give red the vector [1, 0] and blue the vector [0, 1]. The stream red, blue, red gives three rows and two columns. Rows one and three represent different occurrences of the same color.

**Formal notation.** X has shape T × d. T counts positions and d counts numbers per token. X[i] is the vector at position i. Python starts counting at zero.

**Predict before you run.** If a fourth signal is blue, which dimension changes?

In [1]:
import torch
colors = {"red": [1., 0.], "blue": [0., 1.]}
stream = ["red", "blue", "red"]
X = torch.tensor([colors[token] for token in stream])
print(X.tolist())
print(tuple(X.shape))

[[1.0, 0.0], [0.0, 1.0], [1.0, 0.0]]
(3, 2)


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The number of positions grows from three to four. Each color still has two coordinates, so the shape becomes 4 × 2.

</details>

## Ask one question of all rows

A query is a vector describing what a position seeks. A key is a vector offered by a candidate position for comparison. Multiply matching coordinates and sum them to get a score. We choose these vectors by hand first so the comparison is visible.

Take the last red query [1, 0]. Its similarity scores against the three color keys are 1, 0, 1. The result gives one score per position. The two reds tie.

**Formal notation.** sⱼ = q · kⱼ. Here j names a candidate, q is one query and kⱼ is candidate j’s key. In code, K @ q computes all three scores.

**Predict before you run.** What changes if the query becomes [0, 1]?

In [2]:
import torch
q = torch.tensor([1., 0.])
K = torch.tensor([[1., 0.], [0., 1.], [1., 0.]])
print((K @ q).tolist())

[1.0, 0.0, 1.0]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The scores become [0, 1, 0]. This query picks out the blue coordinate, which is largest in the middle row.

</details>

## Turn scores into shares

A weighted average needs nonnegative shares that sum to one. Softmax exponentiates every score and divides by the row’s total. Scaled dot-product attention divides the scores by the square root of key width first. This controls growth in score magnitude as the number of coordinates increases.

With two coordinates, divide [1, 0, 1] by √2. The exponentials are about [2.028, 1, 2.028]. Their sum is about 5.056, giving shares [0.401, 0.198, 0.401].

**Formal notation.** aⱼ = exp(sⱼ/√dₖ) / Σₗ exp(sₗ/√dₖ). The symbol Σ adds across candidate positions l. The key width dₖ is 2 here.

**Predict before you run.** Three equal scores enter the softmax. How much weight does each get?

In [3]:
import torch, math
scores = torch.tensor([1., 0., 1.]) / math.sqrt(2)
a = scores.softmax(dim=0)
print([round(float(x), 3) for x in a])
print(round(float(a.sum()), 3))

[0.401, 0.198, 0.401]
1.0


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Their exponentials are identical. Each receives one third of their total, even if all three original scores are negative.

</details>

## Collect the values

A value is the information a candidate can contribute. The key decides how well a candidate matches. Its value enters the output. Multiply each value by its share and sum. A large share makes the corresponding value more influential.

Suppose three signals carry brightness readings [8, 2, 6]. A red query yields shares near [0.401, 0.198, 0.401]. Its output brightness is about 6.01. Even two matching colors can contribute different measurements.

**Formal notation.** o = Σⱼ aⱼvⱼ. Each value vⱼ can be a single measurement or a vector. The result o has the same number of coordinates as one value.

**Predict before you run.** If all three brightness values are 4, what comes out?

In [4]:
import math, torch
scores = torch.tensor([1., 0., 1.]) / math.sqrt(2)
brightness = torch.tensor([8., 2., 6.])
print(round(float(scores.softmax(0) @ brightness), 3))

6.011


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The output is 4. Multiplying any shares that sum to one by the same value gives that value.

</details>

## Compute every query at once

So far only one position asked for information. Stack all queries into a matrix Q and compare every row with every key. Each row of the score matrix contains the candidates for one query. A heatmap gives large weights a stronger color.

With red, blue, red, the middle row favors the middle key. The first and last rows have identical weights in this content-only calculation. Right-multiplying the weights by the values returns one mixture per query.

**Formal notation.** S = QKᵀ/√dₖ, A = softmax(S) across columns, O = AV. The transpose ᵀ turns key rows into columns. If V is T × 1, O is T × 1.

**Predict before you run.** Which row and column give the weight from the middle query to the final red?

In [5]:
import torch, math
Q = K = torch.tensor([[1., 0.], [0., 1.], [1., 0.]])
V = torch.tensor([[8.], [2.], [6.]])
A = (Q @ K.T / math.sqrt(K.shape[1])).softmax(dim=-1)
print([[round(float(x), 3) for x in row] for row in A])
print([round(float(x), 3) for x in (A @ V).flatten()])

[[0.401, 0.198, 0.401], [0.248, 0.503, 0.248], [0.401, 0.198, 0.401]]
[6.011, 4.483, 6.011]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Use the second row and third column, A[1, 2] in Python. Rows ask the questions and columns identify the candidate keys.

</details>

## Give order a numerical representation

Without position information, shuffling the key and value rows together simply shuffles the terms of a sum for a fixed query. To make position available, attach a position vector to the token embedding before constructing queries and keys. Real models may learn these position vectors.

For a one-coordinate illustration, assign red 1 and blue 0. Add position offsets 0, 0.5 and 1. The two reds now have representations 1 and 2. A fixed positive query can assign them different scores.

**Formal notation.** hᵢ = eᵢ + pᵢ. The token vector eᵢ and position vector pᵢ must have matching shapes. The example uses one number each.

**Predict before you run.** Does adding position information force the newest token to receive the most weight?

In [6]:
import torch
tokens = torch.tensor([1., 0., 1.])
position = torch.tensor([0., 0.5, 1.])
for keys in [tokens, tokens + position]:
    print([round(float(w), 3) for w in keys.softmax(0)])

[0.422, 0.155, 0.422]
[0.231, 0.14, 0.629]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

No. This example chose offsets and a positive query that favor later positions. Learned projections can use or ignore those coordinates.

</details>

## Hide future positions

To predict the next signal after a prefix, that prefix must be all the model sees. A causal mask permits keys up to and including the query position. Set the disallowed scores to negative infinity before softmax. Their exponentials become zero.

For query position two, keys one and two are allowed. Key three receives zero weight. The other weights are recomputed over the allowed entries and still sum to one.

**Formal notation.** Mᵢⱼ = 0 when j ≤ i and −∞ when j > i. A = softmax(S + M) across columns. Position i will predict the token at i + 1.

**Predict before you run.** Can the second position see its own token during next-token training?

In [7]:
import torch, math
X = torch.tensor([[1., 0.], [0., 1.], [1., 0.]])
scores = X @ X.T / math.sqrt(2)
future = torch.ones(3, 3, dtype=torch.bool).triu(diagonal=1)
A = scores.masked_fill(future, -torch.inf).softmax(-1)
print([[round(float(w), 3) for w in row] for row in A])

[[1.0, 0.0, 0.0], [0.33, 0.67, 0.0], [0.401, 0.198, 0.401]]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Yes. It uses tokens one and two to predict token three. It would leak the answer if it could see token three.

</details>

## Build a trainable attention head

In a trainable model, each row of X is transformed three ways. Separate weight matrices learn how to ask, match and transmit information. Wrap these operations in a PyTorch module so an optimizer can update the matrices using a task loss.

For this forward pass, start each projection at the identity matrix so the result can be checked against our earlier color example. Unlike a scalar brightness value, the output here mixes two-coordinate color vectors.

**Formal notation.** Q = XWq, K = XWk, V = XWv, with one trainable matrix per role. PyTorch Linear stores its weight transposed relative to this notation.

**Predict before you run.** What makes this single-head layer self-attention?

In [8]:
import torch, math
from torch import nn
class Head(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.q, self.k, self.v = [nn.Linear(width, width, bias=False) for _ in range(3)]
        with torch.no_grad():
            for layer in (self.q, self.k, self.v):
                layer.weight.copy_(torch.eye(width))
    def forward(self, x):
        scores = self.q(x) @ self.k(x).T / math.sqrt(x.shape[-1])
        return scores.softmax(-1) @ self.v(x)
x = torch.tensor([[1., 0.], [0., 1.], [1., 0.]])
head = Head(2)
out = head(x)
print(tuple(out.shape), [round(float(v.detach()), 3) for v in out[0]])

(3, 2) [0.802, 0.198]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The same input sequence supplies every query, key and value. Each position draws information from positions in that sequence.

</details>

## Assemble a transformer block

Multiple heads allow different learned comparisons at the same position. Concatenate their output vectors and project back to the input width. A transformer block combines that attention update with the residual connections and small feedforward network from Modules 08–09. Layer normalization rescales coordinates within a token.

Build a two-head encoder block of width four. Each head has two coordinates. Three input positions produce three output positions. We use fixed example inputs and random initial weights here to inspect shape; useful representations require training.

**Formal notation.** One common ordering is H = norm(X + attention(X)), then Y = norm(H + FFN(H)). FFN is a feedforward network applied independently to every token row.

**Predict before you run.** If attention acts across positions, which part of this block can transform each position separately?

In [9]:
import torch
torch.manual_seed(4)
block = torch.nn.TransformerEncoderLayer(d_model=4, nhead=2,
    dim_feedforward=8, dropout=0.0, batch_first=True)
x = torch.tensor([[[1.,0.,0.,0.], [0.,1.,0.,0.], [1.,0.,0.,1.]]])
y = block(x)
print("batch, positions, coordinates:", tuple(y.shape))
print("all finite:", bool(torch.isfinite(y).all()))

batch, positions, coordinates: (1, 3, 4)
all finite: True


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The feedforward network uses the same learned transformation on each row. Its input already contains information mixed across positions by attention.

</details>

## Exercise 1

A one-coordinate query is 1 and two keys are 0 and ln(3). Calculate the attention shares.

<details><summary>Hint</summary>

The scaling factor is √1. Exponentiate both scores before dividing.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The exponentials are 1 and 3. Their sum is 4, so the shares are 1/4 and 3/4.

</details>

## Exercise 2

Use shares 1/4 and 3/4 with value vectors [4, 0] and [0, 8]. Compute the output.

<details><summary>Hint</summary>

Apply the same shares separately to each coordinate.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The coordinates are (1/4)4 + (3/4)0 = 1 and (1/4)0 + (3/4)8 = 6. The output is [1, 6].

</details>

## Exercise 3

All scores are zero in a four-token stream. What is the third row of causal attention?

<details><summary>Hint</summary>

Which keys have been seen by position three?

</details>

Check your work.

<details><summary>Reveal answer</summary>

Three keys are allowed and tie. Row three is [1/3, 1/3, 1/3, 0].

</details>

## Exercise 4

A sequence has five positions, queries and keys have width two, and values have width three. Give the score and output shapes.

<details><summary>Hint</summary>

First count query–key pairs, then count coordinates in a mixture of values.

</details>

Check your work.

<details><summary>Reveal answer</summary>

Each of five queries compares with five keys, so the score shape is 5 × 5. Each query produces three output coordinates, giving 5 × 3.

</details>

## Exercise 5

A fixed query attends to two tokens with no positions or mask. Swap their keys and values together. Explain the result.

<details><summary>Hint</summary>

Write the two weighted terms before swapping.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The two scores swap, their softmax weights swap and the same two weighted values are added in reverse order. The mixture is unchanged. A position-dependent key or a causal mask can break this symmetry.

</details>

## Exercise 6

A student multiplies probabilities by a causal mask after softmax and gets an allowed row sum of 0.6. Repair the calculation.

<details><summary>Hint</summary>

When should the normalizing denominator be computed?

</details>

Check your work.

<details><summary>Reveal answer</summary>

Mask the future scores before softmax. The denominator then sums only the permitted exponentials and the allowed weights sum to one.

</details>

The last row now holds a vector built from the prefix. Module 12 turns that vector into one score per possible next token, then uses the actual next token to teach the model.

A vocabulary contains four symbols. What must we compute from the final context vector before predicting the next symbol?

<details><summary>Reveal answer</summary>

Produce four scores, one per symbol, then normalize them into a probability distribution. That distribution lets us score the symbol that actually follows.

</details>